# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 06: Validación Estricta, Constraints y Params Query/Body
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Filtro manual de rangos y longitudes de texto*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Validaciones Pydantic Field(ge=18, min_length=3)* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de desbordamiento de cadenas y números negativos*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion06.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Nov 18 2026 23:42:07) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Filtro manual de rangos y longitudes de texto

In [3]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano")

class PayloadRequest(BaseModel):
    dato: str

# TODO (El Artesano): Implementar lógica manual del endpoint
@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}


Overwriting app_artesano.py


In [5]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano")

class PayloadRequest(BaseModel):
    dato: str
    edad: int

@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    texto_limpio = payload.dato.strip()
    if not texto_limpio:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="El campo 'dato' no puede estar vacío"
        )
    if len(texto_limpio) < 3 or len(texto_limpio) > 50:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="El campo 'dato' debe tener entre 3 y 50 caracteres"
        )
    if payload.edad < 18:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="El campo 'edad' debe ser mayor o igual a 18 (no se permiten menores ni negativos)"
        )
    return {
        "status": "success",
        "mensaje": f"Procesado: {texto_limpio}",
        "edad": payload.edad
    }

Overwriting app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [7]:
%%writefile app_ia.py
from fastapi import FastAPI

%%writefile app_ia.py
from fastapi import FastAPI, Query
from pydantic import BaseModel, Field

app = FastAPI(title="API Cientifico - Pydantic")

class PayloadEstricta(BaseModel):
    dato: str = Field(..., min_length=3, max_length=50, description="Texto entre 3 y 50 caracteres")
    edad: int = Field(..., ge=18, description="Entero mayor o igual a 18")

@app.post("/api/recurso")
def procesar_recurso_ia(
    payload: PayloadEstricta,
    etiqueta: str = Query(default="general", min_length=3, max_length=20)
):
    limpio = payload.dato.strip()
    return {
        "status": "success",
        "etiqueta": etiqueta,
        "dato": limpio,
        "edad": payload.edad
    }


Overwriting app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de desbordamiento de cadenas y números negativos

In [9]:
import threading
import time
import uvicorn
import httpx
from app_artesano import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 06 ===\n")
r = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"Respuesta Servidor: HTTP {r.status_code} - {r.json()}")
assert r.status_code == 200
print("PASO ✅ - Test Harness HTTP ejecutado con exito.")


SyntaxError: invalid syntax (app_ia.py, line 3)